# 第 2 天练习 —— 本地 Ollama 网页摘要

## 练习目标（理念）

把第 1 天的「网页抓取 → LLM 摘要」流程，从 **OpenAI 云端** 换成 **Ollama 本地开源模型**：

- **输入**：任意网页 URL
- **处理**：用 `requests` + BeautifulSoup 抽出正文，再喂给本地模型
- **输出**：在笔记本里用 Markdown 展示摘要

后续项目若想**避免付费 API**、或希望**数据不离开本机**，可以复用同一套「OpenAI 兼容客户端 + `base_url` 指向本地」写法。

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 网页抓取 | `requests.get` + `BeautifulSoup.get_text` |
| Chat Completions | `ollama.chat.completions.create(...)` |
| `messages`（system / user） | system 定「怎么摘要」，user 放网页正文 |
| OpenAI Python SDK | 仍用 `OpenAI(...)`，但 `base_url` 指向 Ollama |
| 本地开源模型 | `llama3.2`（也可改用更小的 `llama3.2:1b`） |

## 怎么跑

1. 安装并启动 [Ollama](https://ollama.com)；浏览器打开 [http://localhost:11434/](http://localhost:11434/) 应看到 `Ollama is running`
2. 若服务未起：新开终端执行 `ollama serve`；再 `ollama pull llama3.2`（本机很慢可用 `llama3.2:1b`）
3. 从上到下依次运行单元格；最后一格会摘要 `https://edwarddonner.com`
4. 注意：下面「拉取」单元格用的是 `llama3.2:1b`，而调用处默认 `model="llama3.2"`——两者需与本机已安装模型名一致，否则按本机情况二选一对齐

**优点：** 无 API 费用；数据留在本地。  
**缺点：** 能力通常弱于前沿闭源模型；本机 CPU/GPU 会决定速度。


In [ ]:
# ========== 导入：网页抓取 + 本地 LLM 客户端 ==========

# 导入标准库请求库 requests：用 HTTP GET 下载网页 HTML
import requests
# 从 bs4 导入 BeautifulSoup：把 HTML 解析成可抽取文本的文档树
from bs4 import BeautifulSoup
# 从 IPython.display 导入 Markdown、display：在笔记本里漂亮渲染摘要
from IPython.display import Markdown, display
# 从 openai 导入 OpenAI：官方 SDK；这里会把它的 base_url 指到本地 Ollama（OpenAI-compatible）
from openai import OpenAI


In [ ]:
# ========== 客户端：OpenAI SDK → 本地 Ollama（兼容 /v1） ==========

# Ollama 提供的 OpenAI 兼容基址：端口 11434，路径 /v1（不是原生 /api/chat）
OLLAMA_BASE_URL = "http://localhost:11434/v1"

# 创建客户端：base_url 指向本机；api_key 对 Ollama 无实际校验，但 SDK 要求非空，故用占位 "ollama"
ollama = OpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key="ollama"  # 必填但可为占位值
)


In [ ]:
# ========== 健康检查：确认本机 Ollama 服务已起来 ==========

# GET 根地址 http://localhost:11434；若正常，响应体多为 b'Ollama is running'
# .content 取出原始字节，便于在输出里一眼确认服务状态
requests.get("http://localhost:11434").content


In [ ]:
# ========== 拉取模型：第一次用需下载权重到本机 ==========

# Jupyter/IPython 的 shell magic：在笔记本里执行终端命令 ollama pull
# llama3.2:1b 是更小的 1B 变体，本机慢时更友好；需与后面 chat 的 model= 字符串对齐
!ollama pull llama3.2:1b


In [ ]:
# ========== 抓取：URL → 纯文本（截断以加快本地推理） ==========

def fetch_website_contents(url):
    # 入参 url：要摘要的网页地址（字符串）
    # GET 下载整页 HTML；默认不设超时（保持原逻辑）
    response = requests.get(url)
    # 用 html.parser 解析响应文本，得到 BeautifulSoup 对象
    soup = BeautifulSoup(response.text, "html.parser")

    # 抽出可见文本；separator="\n" 让块与块之间换行，可读性更好
    text = soup.get_text(separator="\n")

    # 只返回前 5000 字符：本地小模型上下文/速度有限，截断是刻意折中（不是丢弃逻辑）
    return text[:5000]  # 为提速做截断


In [ ]:
# ========== 组消息：system 定角色，user 塞网页正文 ==========

def build_messages(website_text):
    # 返回 OpenAI/Ollama Chat Completions 所需的 messages 列表
    return [
        {
            # system：约束模型「怎么摘要」（清晰、简洁）；prompt 字符串保持英文可运行
            "role": "system",
            "content": "You are a helpful assistant that summarizes websites clearly and concisely."
        },
        {
            # user：把抓到的网页文本拼进提示；f-string 插入 website_text
            "role": "user",
            "content": f"Summarize this website:\n\n{website_text}"
        }
    ]


In [ ]:
# ========== 摘要主流程：抓取 → 组消息 → 本地 chat.completions ==========

def summarize_with_ollama(url):
    # 先抓网页正文（可能已截断到 5000 字）
    website = fetch_website_contents(url)
    # 再打成 system/user 两条 messages
    messages = build_messages(website)

    # 走前面创建的 ollama 客户端；model 名须与本机 ollama list 一致
    response = ollama.chat.completions.create(
        model="llama3.2",
        messages=messages
    )

    # 取第一条 choice 的 message.content，即模型生成的摘要字符串
    return response.choices[0].message.content


In [ ]:
# ========== 展示：把摘要渲染成笔记本 Markdown ==========

def display_summary(url):
    # 调用上面的端到端函数拿到纯文本摘要
    summary = summarize_with_ollama(url)
    # display(Markdown(...))：在 Jupyter 里按 Markdown 排版显示，而不是原始字符串
    display(Markdown(summary))


In [ ]:
# ========== 试跑：对课程作者站点做一次本地摘要 ==========

# 传入目标 URL；内部会抓取 edwarddonner.com 并用本地 llama3.2 摘要
# 若报模型不存在：先 ollama pull，或把 summarize_with_ollama 里的 model 改成你已安装的名字
display_summary("https://edwarddonner.com")
